## PASO 1 — Cargar librerías y datos

### ¿Qué es cada librería y para qué la usamos?

- **pandas** → la herramienta principal para trabajar con tablas de datos (como Excel pero en código)
- **numpy** → cálculos matemáticos rápidos; pandas lo usa internamente
- **matplotlib / seaborn** → para crear gráficos. Seaborn hace gráficos más bonitos con menos código
- **warnings** → para silenciar mensajes de advertencia que no nos interesan

> 💡 **Tip de entrevista:** Si te preguntan *"¿qué librería usas para análisis de datos en Python?"*, la respuesta es **Pandas**. Es el estándar de la industria.

# 02 — Data Cleaning & EDA
## Solvix E-commerce Analytics

Este notebook tiene dos objetivos:
1. **Limpiar** los datos crudos para que sean confiables y consistentes
2. **Explorar** los datos visualmente para descubrir patrones de negocio

---

### ¿Por qué limpiar antes de analizar?

Cuando recibes datos reales (de un ERP, de Shopify, de Meta Ads), casi siempre vienen con problemas:
- Fechas guardadas como texto
- Columnas numéricas con caracteres raros
- Filas duplicadas por errores de exportación
- Valores imposibles (precio negativo, cantidad = 0)

Si analizas datos sucios, tus conclusiones serán incorrectas. **Basura entra, basura sale.**

---

### El proceso que seguiremos:

| Paso | Qué hacemos | Por qué importa |
|---|---|---|
| 1 | Cargar y revisar estructura | Entender qué tenemos |
| 2 | Buscar nulos y duplicados | Detectar problemas |
| 3 | Corregir tipos de datos | Fechas, números, categorías |
| 4 | Validar rangos y consistencia | Detectar valores imposibles |
| 5 | Exportar datos limpios | Base para el análisis |

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.style.use('ggplot')

print("✓ Librerías cargadas correctamente")

✓ Librerías cargadas correctamente


### Cargar los archivos CSV

`pd.read_csv()` lee un archivo CSV y lo convierte en un **DataFrame** — que es básicamente una tabla con filas y columnas, igual que una hoja de Excel.

La ruta `../data/raw/` significa: *"sube una carpeta desde notebooks/ y entra a data/raw/"*

In [2]:
df_ordenes = pd.read_csv('../data/raw/solvix_ordenes_raw.csv')
df_ads     = pd.read_csv('../data/raw/solvix_meta_ads_raw.csv')

print(f"Órdenes:  {df_ordenes.shape[0]:,} filas  x  {df_ordenes.shape[1]} columnas")
print(f"Meta Ads: {df_ads.shape[0]:,} filas  x  {df_ads.shape[1]} columnas")

Órdenes:  3,500 filas  x  14 columnas
Meta Ads: 360 filas  x  5 columnas


### Ver las primeras filas

`.head()` muestra las primeras 5 filas del DataFrame. Es lo primero que hace cualquier analista al recibir datos nuevos — ver *cómo luce* la data antes de hacer cualquier cálculo.

> 💡 También existe `.tail()` para ver las últimas filas, y `.sample(5)` para ver 5 filas aleatorias.

In [3]:
print("=== ÓRDENES - Primeras 3 filas ===")
df_ordenes.head(3)

=== ÓRDENES - Primeras 3 filas ===


In [4]:
print("=== META ADS - Primeras 3 filas ===")
df_ads.head(3)

=== META ADS - Primeras 3 filas ===


## PASO 2 — Diagnóstico: tipos de datos y estructura

### ¿Qué es `.info()`?

Es el **examen médico** del DataFrame. Te dice:
- El nombre de cada columna
- Cuántos valores NO nulos tiene (si dice 3500/3500 → no hay nulos)
- El **tipo de dato** que Pandas le asignó automáticamente

### Tipos de datos más comunes:

| Tipo en Pandas | Qué significa |
|---|---|
| `object` | Texto (strings) |
| `int64` | Número entero |
| `float64` | Número decimal |
| `datetime64` | Fecha y hora |
| `bool` | Verdadero/Falso |

> ⚠️ **El problema más común:** Pandas lee las fechas como `object` (texto) al importar un CSV. Hay que convertirlas manualmente a `datetime64`. Si no lo haces, no puedes filtrar por mes, calcular días entre fechas, etc.

In [5]:
print("=== ÓRDENES — Estructura y tipos ===")
df_ordenes.info()

=== ÓRDENES — Estructura y tipos ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID_Orden            3500 non-null   object 
 1   Fecha               3500 non-null   object 
 2   ID_Cliente          3500 non-null   object 
 3   ID_Producto         3500 non-null   object 
 4   Nombre_Producto     3500 non-null   object 
 5   Categoria           3500 non-null   object 
 6   Cantidad            3500 non-null   int64  
 7   Ingreso_Total       3500 non-null   int64  
 8   COGS_Total          3500 non-null   int64  
 9   Costo_Envio         3500 non-null   float64
 10  Ganancia_Bruta      3500 non-null   float64
 11  Ciudad_Destino      3500 non-null   object 
 12  Fuente_Adquisicion  3500 non-null   object 
 13  Estado_Envio        3500 non-null   object 
dtypes: float64(2), int64(3), object(9)
memory usage: 382.9+ KB


In [6]:
print("=== META ADS — Estructura y tipos ===")
df_ads.info()

=== META ADS — Estructura y tipos ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Fecha               360 non-null    object 
 1   Nombre_Campaña      360 non-null    object 
 2   Gasto_Diario_USD    360 non-null    float64
 3   Clics               360 non-null    int64  
 4   Compras_Atribuidas  360 non-null    int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 14.2+ KB


## PASO 3 — Buscar nulos y duplicados

### ¿Qué es un valor nulo?

Un valor nulo (también llamado `NaN` — *Not a Number*) es una celda **vacía**. Puede ocurrir cuando:
- Un cliente no completó un campo del formulario
- Hubo un error en la exportación del sistema
- Un dato no aplica para esa fila

### ¿Por qué son peligrosos?
Si calculas el promedio de una columna con nulos, el resultado puede ser incorrecto. Pandas los ignora en algunos casos y en otros no — hay que decidir qué hacer con ellos: **eliminar la fila**, **rellenar con un valor**, o **dejarlos si no afectan el análisis**.

### ¿Qué es un duplicado?
Una fila que aparece más de una vez. En ventas, un duplicado puede significar que se cobró dos veces al cliente o que hubo un error al exportar el reporte.

In [7]:
# .isnull() devuelve True/False por cada celda
# .sum() cuenta cuántos True hay por columna
print("=== NULOS EN ÓRDENES ===")
nulos_ordenes = df_ordenes.isnull().sum()
print(nulos_ordenes)
print(f"\nTotal nulos: {nulos_ordenes.sum()}")

print("\n=== NULOS EN META ADS ===")
nulos_ads = df_ads.isnull().sum()
print(nulos_ads)
print(f"\nTotal nulos: {nulos_ads.sum()}")

=== NULOS EN ÓRDENES ===
ID_Orden              0
Fecha                 0
ID_Cliente            0
ID_Producto           0
Nombre_Producto       0
Categoria             0
Cantidad              0
Ingreso_Total         0
COGS_Total            0
Costo_Envio           0
Ganancia_Bruta        0
Ciudad_Destino        0
Fuente_Adquisicion    0
Estado_Envio          0
dtype: int64

Total nulos: 0

=== NULOS EN META ADS ===
Fecha                 0
Nombre_Campaña        0
Gasto_Diario_USD      0
Clics                 0
Compras_Atribuidas    0
dtype: int64

Total nulos: 0


In [8]:
# .duplicated() marca True en filas que ya aparecieron antes
# .sum() cuenta cuántas hay
print("=== DUPLICADOS ===")
dup_ordenes = df_ordenes.duplicated().sum()
dup_ads     = df_ads.duplicated().sum()

print(f"Filas duplicadas en Órdenes:  {dup_ordenes}")
print(f"Filas duplicadas en Meta Ads: {dup_ads}")

# También verificamos que los ID de orden sean únicos
ids_unicos = df_ordenes['ID_Orden'].nunique()
print(f"\nIDs de orden únicos: {ids_unicos} de {len(df_ordenes)} total")
print("→ ¿Hay IDs repetidos?", "SÍ ⚠️" if ids_unicos < len(df_ordenes) else "NO ✓")

=== DUPLICADOS ===
Filas duplicadas en Órdenes:  0
Filas duplicadas en Meta Ads: 0

IDs de orden únicos: 3500 de 3500 total
→ ¿Hay IDs repetidos? NO ✓


## PASO 4 — Corregir tipos de datos

### El problema con las fechas

Cuando Pandas lee un CSV, la columna `Fecha` llega como texto: `"2025-11-03 14:32:00"`.

Para Pandas eso es solo una cadena de caracteres — como cualquier palabra. **No sabe que es una fecha.**

Convertirla a `datetime64` le dice a Pandas: *"esto es tiempo"*. A partir de ahí puedes:
- Extraer el mes: `df['Fecha'].dt.month`
- Extraer el día de la semana: `df['Fecha'].dt.day_name()`
- Filtrar por rango: `df[df['Fecha'] >= '2025-12-01']`
- Agrupar por semana o mes automáticamente

> 💡 **Tip de entrevista:** *"¿Cómo trabajas con fechas en Pandas?"* — Respuesta: *"Primero convierto la columna con `pd.to_datetime()`, y luego uso el accessor `.dt` para extraer componentes como mes, semana o día de la semana."*

In [9]:
print("Tipo ANTES de convertir:", df_ordenes['Fecha'].dtype)

# pd.to_datetime() convierte texto → fecha
df_ordenes['Fecha'] = pd.to_datetime(df_ordenes['Fecha'])
df_ads['Fecha']     = pd.to_datetime(df_ads['Fecha'])

print("Tipo DESPUÉS de convertir:", df_ordenes['Fecha'].dtype)

# Ahora podemos extraer componentes útiles
df_ordenes['Mes']          = df_ordenes['Fecha'].dt.month
df_ordenes['Nombre_Mes']   = df_ordenes['Fecha'].dt.strftime('%B')   # Ej: "November"
df_ordenes['Dia_Semana']   = df_ordenes['Fecha'].dt.day_name()        # Ej: "Monday"
df_ordenes['Semana_Anio']  = df_ordenes['Fecha'].dt.isocalendar().week.astype(int)

print("\n✓ Columnas de tiempo agregadas: Mes, Nombre_Mes, Dia_Semana, Semana_Anio")
print("\nEjemplo de las nuevas columnas:")
df_ordenes[['Fecha','Mes','Nombre_Mes','Dia_Semana','Semana_Anio']].head(4)

Tipo ANTES de convertir: object
Tipo DESPUÉS de convertir: datetime64[ns]

✓ Columnas de tiempo agregadas: Mes, Nombre_Mes, Dia_Semana, Semana_Anio

Ejemplo de las nuevas columnas:


## PASO 5 — Validar rangos y consistencia

### ¿Qué validamos aquí?

Aunque los datos vengan sin nulos ni duplicados, pueden tener **valores imposibles o inconsistentes**:

- ¿Hay órdenes con `Ingreso_Total = 0` o negativo?
- ¿Hay `Cantidad = 0`?
- ¿Hay fechas fuera del período esperado (nov 2025 – abr 2026)?
- ¿Los valores de `Estado_Envio` y `Ciudad_Destino` son los esperados (sin errores de tipeo)?

`.describe()` te da un resumen estadístico de las columnas numéricas: mínimo, máximo, promedio, percentiles. Con eso detectas valores raros de un vistazo.

In [10]:
# Resumen estadístico de columnas numéricas
print("=== ESTADÍSTICAS ÓRDENES ===")
df_ordenes[['Cantidad','Ingreso_Total','COGS_Total','Costo_Envio','Ganancia_Bruta']].describe().round(2)

=== ESTADÍSTICAS ÓRDENES ===


In [11]:
# Verificar rango de fechas
print("=== RANGO DE FECHAS ===")
print(f"Fecha más antigua: {df_ordenes['Fecha'].min()}")
print(f"Fecha más reciente: {df_ordenes['Fecha'].max()}")
print(f"Período esperado:  Nov 2025 → Abr 2026")

# Verificar valores categóricos (buscar errores de tipeo)
print("\n=== VALORES ÚNICOS — Estado_Envio ===")
print(df_ordenes['Estado_Envio'].value_counts())

print("\n=== VALORES ÚNICOS — Ciudad_Destino ===")
print(df_ordenes['Ciudad_Destino'].value_counts())

print("\n=== VALORES ÚNICOS — Fuente_Adquisicion ===")
print(df_ordenes['Fuente_Adquisicion'].value_counts())

=== RANGO DE FECHAS ===
Fecha más antigua: 2025-11-01 01:17:00
Fecha más reciente: 2026-04-30 23:28:00
Período esperado:  Nov 2025 → Abr 2026

=== VALORES ÚNICOS — Estado_Envio ===
Estado_Envio
Entregado      2893
En tránsito     387
Devuelto        220
Name: count, dtype: int64

=== VALORES ÚNICOS — Ciudad_Destino ===
Ciudad_Destino
Bogotá          1412
Medellín         903
Cali             495
Barranquilla     346
Bucaramanga      344
Name: count, dtype: int64

=== VALORES ÚNICOS — Fuente_Adquisicion ===
Fuente_Adquisicion
Campaña_Vaso_Vertical    1894
Campaña_Bici_Carrusel     704
Orgánico_Instagram        543
Directo                   359
Name: count, dtype: int64


## PASO 6 — Exportar datos limpios

### ¿Por qué separar raw de processed?

- `data/raw/` → los datos **tal como llegaron**, nunca se tocan. Son el respaldo original.
- `data/processed/` → los datos **después de la limpieza**. Estos son los que usarán SQL y Power BI.

Esta separación es una buena práctica estándar en proyectos de datos. Si algo sale mal en el análisis, siempre puedes volver al raw y empezar de nuevo.

> 💡 **En una empresa real:** el `raw` vendría de un sistema (Shopify, SAP, HubSpot). Tú nunca modificas esa fuente — solo trabajas con una copia procesada.

In [12]:
import os
os.makedirs('../data/processed', exist_ok=True)

df_ordenes.to_csv('../data/processed/solvix_ordenes_clean.csv', index=False)
df_ads.to_csv('../data/processed/solvix_ads_clean.csv',         index=False)

print("✓ Archivos exportados a data/processed/")
print(f"  → solvix_ordenes_clean.csv  ({len(df_ordenes):,} filas, {len(df_ordenes.columns)} columnas)")
print(f"  → solvix_ads_clean.csv      ({len(df_ads):,} filas, {len(df_ads.columns)} columnas)")
print(f"\nColumnas finales en órdenes:")
print(list(df_ordenes.columns))

✓ Archivos exportados a data/processed/
  → solvix_ordenes_clean.csv  (3,500 filas, 18 columnas)
  → solvix_ads_clean.csv      (360 filas, 5 columnas)

Columnas finales en órdenes:
['ID_Orden', 'Fecha', 'ID_Cliente', 'ID_Producto', 'Nombre_Producto', 'Categoria', 'Cantidad', 'Ingreso_Total', 'COGS_Total', 'Costo_Envio', 'Ganancia_Bruta', 'Ciudad_Destino', 'Fuente_Adquisicion', 'Estado_Envio', 'Mes', 'Nombre_Mes', 'Dia_Semana', 'Semana_Anio']
